# Tau-monitor — Phi-loven P10

Måler tau = r_eff / r_max for språkmodeller per lag.

**Goldilocks-intervallet:** [tau_min, tau_max] = [exp(-γ), 1/ζ(3)] ≈ [0.5615, 0.8319]

- `tau < tau_min` → systemet kollapser mot lavdimensjonal degenerasjon
- `tau ∈ [tau_min, tau_max]` → Goldilocks: systemet prosesserer genuint ny informasjon
- `tau > tau_max` → beholderen er full, systemet gjengir heller enn å oppdage

**Del 1:** GPT-2 (ingen autentisering)

**Del 2:** Microsoft Phi-2 (2.7B parametere, ingen lisens nødvendig)

In [ ]:
!pip install transformers torch matplotlib huggingface_hub -q

In [ ]:
import torch, math, numpy as np, matplotlib.pyplot as plt
from transformers import AutoTokenizer, AutoModel

EULER_MASCHERONI = 0.5772156649015328
APERY            = 1.2020569031595942
TAU_MIN = math.exp(-EULER_MASCHERONI)
TAU_MAX = 1.0 / APERY
ALPHA   = 0.42

print(f"Goldilocks: [{TAU_MIN:.4f}, {TAU_MAX:.4f}]")

def compute_tau(hidden_state):
    H = hidden_state.float()
    N, d = H.shape
    r_max = min(N, d)
    S = torch.linalg.svdvals(H)
    S_sq = S ** 2
    p = S_sq / (S_sq.sum() + 1e-12)
    H_spectral = -(p * torch.log(p + 1e-12)).sum().item()
    r_eff = math.exp(H_spectral)
    tau   = r_eff / r_max
    return {"tau": tau, "r_eff": r_eff, "r_max": r_max, "goldilocks": TAU_MIN <= tau <= TAU_MAX}

def run_test(model, tokenizer, prompts, title, max_length=256):
    all_results = {}
    print(f"\n{title}")
    print(f"{'Tekst':<15} {'tau (siste lag)':<18} {'r_eff':<10} {'r_max':<10} Status")
    print("-" * 70)
    for label, text in prompts.items():
        inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
        with torch.no_grad():
            outputs = model(**inputs)
        layer_results = []
        for hs in outputs.hidden_states:
            layer_results.append(compute_tau(hs[0]))
        all_results[label] = layer_results
        r = layer_results[-1]
        status = "GOLDILOCKS ✓" if r["goldilocks"] else ("OVER" if r["tau"] > TAU_MAX else "UNDER")
        print(f"{label:<15} {r['tau']:<18.4f} {r['r_eff']:<10.2f} {r['r_max']:<10} {status}")
    return all_results

def plot_results(all_results, title):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    ax = axes[0]
    for label, results in all_results.items():
        ax.plot([r["tau"] for r in results], marker='o', markersize=3, label=label)
    ax.axhspan(TAU_MIN, TAU_MAX, alpha=0.08, color='green')
    ax.axhline(TAU_MIN, color='green', linestyle='--', linewidth=0.8, label=f'tau_min={TAU_MIN:.4f}')
    ax.axhline(TAU_MAX, color='red',   linestyle='--', linewidth=0.8, label=f'tau_max={TAU_MAX:.4f}')
    ax.set_xlabel("Lag"); ax.set_ylabel("tau"); ax.set_title("Tau per lag")
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)
    ax2 = axes[1]
    labels = list(all_results.keys())
    x = np.arange(len(labels))
    ax2.bar(x - 0.175, [all_results[l][-1]["r_eff"] for l in labels], 0.35, label='r_eff', color='steelblue', alpha=0.8)
    ax2.bar(x + 0.175, [all_results[l][-1]["r_max"] for l in labels], 0.35, label='r_max', color='lightcoral', alpha=0.8)
    ax2.set_xticks(x); ax2.set_xticklabels(labels, rotation=15, ha='right')
    ax2.set_title("r_eff vs r_max (siste lag)"); ax2.legend(); ax2.grid(True, alpha=0.3, axis='y')
    plt.suptitle(title, fontsize=13, fontweight='bold')
    plt.tight_layout()
    fname = title.lower().replace(" ", "_").replace("/", "") + ".png"
    plt.savefig(fname, dpi=150, bbox_inches='tight')
    plt.show()
    print(f"Lagret: {fname}")

## Del 1 — GPT-2

In [ ]:
gpt2_tok = AutoTokenizer.from_pretrained("gpt2")
gpt2_mod = AutoModel.from_pretrained("gpt2", output_hidden_states=True)
gpt2_mod.eval()
print(f"Lastet: gpt2 — {sum(p.numel() for p in gpt2_mod.parameters()):,} parametere")

In [ ]:
prompts = {
    "Koherent":   "The relationship between energy and matter was first described by Einstein. "
                  "His equation fundamentally changed our understanding of the universe and "
                  "opened the door to modern physics, quantum mechanics, and nuclear energy.",
    "Repetitivt": "the the the the the the the the the the the the the the the the the the "
                  "the the the the the the the the the the the the the the the the the the",
    "Tilfeldig":  "purple quantum seventeen beneath oscillates mirror forgotten banana "
                  "recursive telescope whisper iron seventeen again cloud orange river",
    "Framleis":   "Identity is not something you have. It is something you do through each "
                  "filtering, each choice about what passes through and what does not. "
                  "What survives change is not the form. It is the selection mechanism."
}

gpt2_results = run_test(gpt2_mod, gpt2_tok, prompts, "Phi-loven tau-monitor — GPT-2")
plot_results(gpt2_results, "Phi-loven tau-monitor — GPT-2")

## Del 2 — Mistral-7B (7B, ingen lisens)

In [ ]:
mistral_tok = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.1")
mistral_mod = AutoModel.from_pretrained("mistralai/Mistral-7B-v0.1", output_hidden_states=True)
mistral_mod.eval()
print(f"Lastet: Mistral-7B — {sum(p.numel() for p in mistral_mod.parameters()):,} parametere")

In [ ]:
long_prompts = {
    "Koherent":   "The relationship between energy and matter was first described by Einstein "
                  "in his famous equation E=mc2. This equation fundamentally changed our "
                  "understanding of the universe and opened the door to modern physics, "
                  "quantum mechanics, nuclear energy, and our understanding of stars. "
                  "The implications extend from the smallest particles to the largest "
                  "structures in the cosmos, connecting mass, energy, and the speed of light "
                  "in a single elegant formula verified countless times by experiment.",
    "Repetitivt": "the the the the the the the the the the the the the the the the the the "
                  "the the the the the the the the the the the the the the the the the the "
                  "the the the the the the the the the the the the the the the the the the",
    "Framleis":   "Identity is not something you have. It is something you do through each "
                  "filtering, each choice about what passes through and what does not. "
                  "What survives change is not the form. It is the selection mechanism. "
                  "Alpha represents the optimal forgetting rate. Without forgetting, the "
                  "system collapses under its own weight. The container becomes too small "
                  "for the reality it tries to hold. Selection is the engine of continued "
                  "existence. What is, is what continues to become. Framleis."
}

mistral_results = run_test(mistral_mod, mistral_tok, long_prompts, "Phi-loven tau-monitor — Mistral-7B", max_length=256)
plot_results(mistral_results, "Phi-loven tau-monitor — Mistral-7B")

## Tolkning

| tau-verdi | Betydning |
|---|---|
| < tau_min (0.5615) | UNDER — systemet kollapser mot lavdimensjonal degenerasjon |
| tau_min–tau_max | GOLDILOCKS — systemet prosesserer genuint ny informasjon |
| > tau_max (0.8319) | OVER — beholderen er full, systemet gjengir heller enn å oppdage |

**Falsifiseringstest:**
- Koherent tekst skal ligge nærmere Goldilocks enn repetitiv tekst
- Llama-3.2:3b skal ha høyere tau enn GPT-2 (større modell = høyere r_eff)
- Hvis begge stemmer — holder loven